# LeetCode 1567 — Maximum Length of Subarray With Positive Product bằng Python

Link đề: https://leetcode.com/problems/maximum-length-of-subarray-with-positive-product/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Greedy

Skeleton LeetCode:

```python
class Solution:
    def getMaxLen(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra **chỉ dấu** của tích là quan trọng: số `0` làm tích bằng 0, số âm đổi dấu, số dương giữ dấu — không cần (và không nên) tính tích thật.
2. Viết baseline **duyệt mọi đoạn con** và theo dõi dấu, `O(n²)`.
3. Xây **quy hoạch động hai trạng thái** `pos[i]`, `neg[i]` = độ dài đoạn dài nhất **kết thúc tại `i`** có tích dương / âm, và chuyển trạng thái đúng cho từng loại phần tử.
4. Giải bằng góc nhìn **tham lam**: chia mảng theo số `0`, trong mỗi đoạn đếm số âm; nếu lẻ thì bỏ phần đầu tới số âm đầu tiên hoặc phần đuôi từ số âm cuối cùng.
5. Tránh các bẫy: cập nhật `pos`/`neg` sai thứ tự, tăng `neg` khi chưa có đoạn âm, quên reset ở số 0.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Tìm **độ dài lớn nhất** của một **đoạn con liên tiếp** mà **tích** tất cả phần tử của nó **dương**. (Đoạn con có thể rỗng — khi đó độ dài là 0.)

**Ví dụ 1:** `nums = [1,-2,-3,4]` → `4`. Cả mảng có tích `24 > 0`.

**Ví dụ 2:** `nums = [0,1,-2,-3,-4]` → `3`. Đoạn dài nhất là `[1,-2,-3]` (tích 6). Không được chứa số 0 vì tích sẽ bằng 0.

**Ví dụ 3:** `nums = [-1,-2,-3,0,1]` → `2`. Đoạn `[-1,-2]` hoặc `[-2,-3]`.

**Ràng buộc:** `1 ≤ nums.length ≤ 10⁵`, `−10⁹ ≤ nums[i] ≤ 10⁹`.

In [1]:
import random
import time
from typing import List

examples = [([1, -2, -3, 4], 4), ([0, 1, -2, -3, -4], 3), ([-1, -2, -3, 0, 1], 2)]
for nums, want in examples:
    print(f"nums = {nums} -> mong đợi {want}")

nums = [1, -2, -3, 4] -> mong đợi 4
nums = [0, 1, -2, -3, -4] -> mong đợi 3
nums = [-1, -2, -3, 0, 1] -> mong đợi 2


## 3. Kiến thức nền

### 3.1 Dấu của tích

- Có số `0` ⇒ tích bằng `0` (không dương).
- Không có `0`: tích dương ⇔ **số lượng số âm là chẵn**.

Hàm `sign(x)` trả `1`, `−1` hoặc `0`; dấu của tích = tích các dấu.

In [2]:
def sign(x):
    return (x > 0) - (x < 0)

for arr in [[1, -2, -3, 4], [1, -2, -3, -4], [5, 0, 7]]:
    s = 1
    for x in arr:
        s *= sign(x)
    print(arr, "-> dấu của tích:", s, "| số âm:", sum(x < 0 for x in arr))

[1, -2, -3, 4] -> dấu của tích: 1 | số âm: 2
[1, -2, -3, -4] -> dấu của tích: -1 | số âm: 3
[5, 0, 7] -> dấu của tích: 0 | số âm: 0


### 3.2 Vì sao **không** nên tính tích thật

Python có số nguyên lớn tuỳ ý nên không tràn số, nhưng tích của `10⁵` số cỡ `10⁹` có tới ~900 000 chữ số — nhân rất chậm. Chỉ cần dấu.

In [3]:
t0 = time.perf_counter()
p = 1
for _ in range(3000):
    p *= 10**9
print(f"tích 3000 số 10^9 có khoảng {int(p.bit_length() * 0.30103) + 1} chữ số, mất {time.perf_counter() - t0:.4f}s (n = 10^5 sẽ chậm hơn rất nhiều)")

tích 3000 số 10^9 có khoảng 27001 chữ số, mất 0.0068s (n = 10^5 sẽ chậm hơn rất nhiều)


### 3.3 Đoạn con "kết thúc tại `i`" — khuôn quy hoạch động trên mảng

Nhiều bài đoạn con (Kadane, LC 53/152) dùng trạng thái "tốt nhất trong các đoạn **kết thúc đúng tại `i`**", rồi đáp án là max qua mọi `i`. Ở đây ta cần **hai** trạng thái vì số âm biến dương thành âm và ngược lại.

In [4]:
# Kadane nhắc lại: tổng lớn nhất của đoạn con kết thúc tại i
nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
best = cur = nums[0]
for x in nums[1:]:
    cur = max(x, cur + x)
    best = max(best, cur)
print("tổng đoạn con lớn nhất (LC 53):", best)

tổng đoạn con lớn nhất (LC 53): 6


### 3.4 Gán đồng thời bằng tuple

`a, b = b, a + b` tính **toàn bộ vế phải trước** rồi mới gán — tránh dùng giá trị vừa cập nhật. Rất quan trọng khi cập nhật `pos`, `neg` cùng lúc.

In [5]:
pos, neg = 2, 0
x = -5
pos, neg = (neg + 1 if neg else 0), pos + 1      # vế phải dùng pos = 2 cũ
print(pos, neg)

0 3


### 3.5 Tách mảng theo số 0

Duyệt và cắt mỗi khi gặp `0`; mỗi đoạn không chứa `0` được xử lý độc lập.

In [6]:
def split_by_zero(nums):
    segs, cur = [], []
    for x in nums:
        if x == 0:
            if cur:
                segs.append(cur)
            cur = []
        else:
            cur.append(x)
    if cur:
        segs.append(cur)
    return segs

print(split_by_zero([0, 1, -2, -3, -4, 0, 5, 0, 0, -1]))

[[1, -2, -3, -4], [5], [-1]]


## 4. Ý tưởng nền tảng / trực giác

Xét ví dụ 2: `[0, 1, -2, -3, -4]`.

1. Số `0` chặn mọi đoạn ⇒ chỉ xét đoạn `[1, -2, -3, -4]`.
2. Đoạn này có **3** số âm (lẻ) ⇒ cả đoạn có tích âm.
3. Muốn số âm chẵn, phải bỏ đi ít nhất một số âm, và vì đoạn con phải liên tiếp, ta chỉ có thể **cắt bỏ một đầu**:
   - bỏ tiền tố tới hết số âm **đầu tiên**: còn `[-3, -4]` (độ dài 2);
   - bỏ hậu tố từ số âm **cuối cùng**: còn `[1, -2, -3]` (độ dài 3).
4. Đáp án `3`.

Bảng dấu khi đi từ trái sang (trong đoạn `[1,-2,-3,-4]`): độ dài đoạn dài nhất **kết thúc tại** mỗi vị trí có tích dương/âm:

| i | x | dài nhất kết thúc tại i, tích **dương** | tích **âm** |
|---|---|------------------------------------------|-------------|
| 1 | 1 | 1 (`[1]`) | 0 |
| 2 | −2 | 0 | 2 (`[1,−2]`) |
| 3 | −3 | 3 (`[1,−2,−3]`) | 1 (`[−3]`) |
| 4 | −4 | 2 (`[−3,−4]`) | 4 (`[1,−2,−3,−4]`) |

Max của cột "dương" là `3`. Cột "âm" giúp tính cột "dương" ở bước sau (âm × âm = dương).

In [7]:
for seg in split_by_zero([0, 1, -2, -3, -4]):
    negs = [i for i, x in enumerate(seg) if x < 0]
    print("đoạn", seg, "| vị trí số âm:", negs)
    if len(negs) % 2 == 0:
        print("  số âm chẵn -> lấy cả đoạn:", len(seg))
    else:
        print("  lẻ -> bỏ tiền tố:", len(seg) - negs[0] - 1, "| bỏ hậu tố:", negs[-1])

đoạn [1, -2, -3, -4] | vị trí số âm: [1, 2, 3]
  lẻ -> bỏ tiền tố: 2 | bỏ hậu tố: 3


## 5. Cách 1 — Brute Force / baseline: mọi đoạn con, theo dõi dấu

**Ý tưởng.** Với mỗi điểm bắt đầu `l`, kéo dài `r` sang phải và cập nhật dấu tích `s *= sign(nums[r])`. Gặp `0` thì dừng kéo dài (mọi đoạn chứa nó có tích 0). Nếu `s > 0` thì cập nhật đáp án `r − l + 1`.

**Độ phức tạp:** `O(n²)` time, `O(1)` space. Với `n = 10⁵` là `5·10⁹` bước — quá chậm.

In [8]:
def brute_force(nums):
    n = len(nums)
    best = 0
    for l in range(n):
        s = 1
        for r in range(l, n):
            if nums[r] == 0:
                break
            if nums[r] < 0:
                s = -s
            if s > 0:
                best = max(best, r - l + 1)
    return best

for nums, want in examples:
    print(brute_force(nums), "mong đợi", want)

4 mong đợi 4
3 mong đợi 3
2 mong đợi 2


In [9]:
# Nút thắt: bậc hai (mảng không có 0 là trường hợp xấu nhất)
rng = random.Random(0)
for n in [1000, 2000, 4000]:
    nums = [rng.choice([-3, 2, 5, -1]) for _ in range(n)]
    t0 = time.perf_counter()
    brute_force(nums)
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=1000: 0.048s


n=2000: 0.188s


n=4000: 0.769s


**Nút thắt:** khi chuyển từ `r` sang `r + 1`, ta làm lại cho **mọi** `l`. Nhưng thông tin cần cho vị trí `r + 1` chỉ là: "đoạn dương dài nhất và đoạn âm dài nhất **kết thúc tại `r`**" — hai con số.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái.** `pos` = độ dài đoạn dài nhất kết thúc tại vị trí hiện tại có tích **dương** (0 nếu không có); `neg` = tương tự cho tích **âm**.

**Chuyển trạng thái khi thêm phần tử `x`:**
- *Nếu* `x > 0`: nối `x` vào đoạn dương ⇒ vẫn dương: `pos' = pos + 1`. Nối vào đoạn âm ⇒ vẫn âm: `neg' = neg + 1` **nếu `neg > 0`**, ngược lại `neg' = 0` (không có đoạn âm nào để nối; riêng `[x]` là dương).
- *Nếu* `x < 0`: nối vào đoạn âm ⇒ thành dương: `pos' = neg + 1` nếu `neg > 0`, ngược lại `0`. Nối vào đoạn dương (hoặc đứng một mình) ⇒ thành âm: `neg' = pos + 1`.
- *Nếu* `x == 0`: `pos' = neg' = 0` (mọi đoạn chứa 0 có tích 0).

Đáp án = `max` của `pos` qua mọi vị trí.

**Góc nhìn tham lam (đếm số âm).** Trong một đoạn không chứa `0` (từ sau số 0 gần nhất `z` tới `i`): nếu số âm **chẵn** thì cả đoạn `(z, i]` dương ⇒ độ dài `i − z`; nếu **lẻ** thì bỏ phần tới số âm đầu tiên `f` ⇒ độ dài `i − f`. Đi một lượt, giữ `z`, `f`, và số âm — đó là cách 4 ở mục 7.3. (Vì ta xét mọi `i` làm điểm kết thúc, trường hợp "bỏ hậu tố từ số âm cuối" được bao hàm tự động: nó là đoạn kết thúc ngay trước số âm cuối.)

In [10]:
def show_dp(nums):
    pos = neg = 0
    for i, x in enumerate(nums):
        if x > 0:
            pos, neg = pos + 1, (neg + 1 if neg else 0)
        elif x < 0:
            pos, neg = (neg + 1 if neg else 0), pos + 1
        else:
            pos = neg = 0
        print(f"i={i}, x={x:>2}: pos={pos}, neg={neg}")

show_dp([0, 1, -2, -3, -4])

i=0, x= 0: pos=0, neg=0
i=1, x= 1: pos=1, neg=0
i=2, x=-2: pos=0, neg=2
i=3, x=-3: pos=3, neg=1
i=4, x=-4: pos=2, neg=4


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quy hoạch động với mảng `pos[]`, `neg[]`

Bản "sách giáo khoa": lưu cả mảng để dễ in và kiểm tra.

Từng đoạn:
1. Khởi tạo `pos[0]`, `neg[0]` theo dấu của `nums[0]`.
2. Với `i ≥ 1`, áp dụng 3 trường hợp ở mục 6.
3. Đáp án `max(pos)`.

In [11]:
def dp_arrays(nums):
    n = len(nums)
    pos = [0] * n
    neg = [0] * n
    pos[0] = 1 if nums[0] > 0 else 0
    neg[0] = 1 if nums[0] < 0 else 0
    for i in range(1, n):
        x = nums[i]
        if x > 0:
            pos[i] = pos[i - 1] + 1
            neg[i] = neg[i - 1] + 1 if neg[i - 1] else 0
        elif x < 0:
            pos[i] = neg[i - 1] + 1 if neg[i - 1] else 0
            neg[i] = pos[i - 1] + 1
        # x == 0: giữ 0
    return max(pos)

for nums, want in examples:
    print(dp_arrays(nums), "mong đợi", want)

4 mong đợi 4
3 mong đợi 3
2 mong đợi 2


Code nộp LeetCode cho cách 2:

In [12]:
class SolutionDPArray:
    def getMaxLen(self, nums: List[int]) -> int:
        n = len(nums)
        pos, neg = [0] * n, [0] * n
        pos[0] = int(nums[0] > 0)
        neg[0] = int(nums[0] < 0)
        for i in range(1, n):
            if nums[i] > 0:
                pos[i] = pos[i - 1] + 1
                neg[i] = neg[i - 1] + 1 if neg[i - 1] else 0
            elif nums[i] < 0:
                pos[i] = neg[i - 1] + 1 if neg[i - 1] else 0
                neg[i] = pos[i - 1] + 1
        return max(pos)

print([SolutionDPArray().getMaxLen(x) for x, _ in examples])

[4, 3, 2]


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### 7.2 Cách 3 — Quy hoạch động `O(1)` bộ nhớ

Chỉ cần giá trị ở vị trí trước ⇒ hai biến `pos`, `neg`, cập nhật **đồng thời** bằng gán tuple.

In [13]:
class SolutionDP:
    def getMaxLen(self, nums: List[int]) -> int:
        pos = neg = best = 0
        for x in nums:
            if x > 0:
                pos, neg = pos + 1, (neg + 1 if neg else 0)
            elif x < 0:
                pos, neg = (neg + 1 if neg else 0), pos + 1
            else:
                pos = neg = 0
            best = max(best, pos)
        return best

print([SolutionDP().getMaxLen(x) for x, _ in examples])

[4, 3, 2]


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### 7.3 Cách 4 — Một lượt với "số 0 gần nhất" và "số âm đầu tiên"

Giữ:
- `zero` = chỉ số của số `0` gần nhất (ban đầu `−1`),
- `first_neg` = chỉ số số âm đầu tiên **sau** `zero` (ban đầu `None`),
- `negs` = số lượng số âm sau `zero`.

Tại mỗi `i` (khác 0): nếu `negs` chẵn ⇒ đoạn `(zero, i]` dương, độ dài `i − zero`; nếu lẻ ⇒ đoạn `(first_neg, i]` dương, độ dài `i − first_neg`. Gặp `0` thì reset.

In [14]:
class SolutionGreedy:
    def getMaxLen(self, nums: List[int]) -> int:
        best = 0
        zero, first_neg, negs = -1, None, 0
        for i, x in enumerate(nums):
            if x == 0:
                zero, first_neg, negs = i, None, 0
                continue
            if x < 0:
                negs += 1
                if first_neg is None:
                    first_neg = i
            if negs % 2 == 0:
                best = max(best, i - zero)
            else:
                best = max(best, i - first_neg)
        return best

print([SolutionGreedy().getMaxLen(x) for x, _ in examples])

[4, 3, 2]


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### 7.4 Cách 5 — Chia đoạn theo số 0, đếm số âm từng đoạn

Rõ ràng nhất về mặt lý luận (gợi ý của đề): với mỗi đoạn không chứa `0`, độ dài `L`, vị trí số âm đầu `f` và cuối `l` (tính trong đoạn):
- số âm chẵn ⇒ `L`;
- lẻ ⇒ `max(L − f − 1, l)` (bỏ tiền tố `[0..f]` hoặc bỏ hậu tố `[l..L−1]`).

In [15]:
class SolutionSegments:
    def getMaxLen(self, nums: List[int]) -> int:
        best = 0
        start = 0
        n = len(nums)
        for i in range(n + 1):
            if i == n or nums[i] == 0:
                seg = nums[start:i]
                negs = [k for k, x in enumerate(seg) if x < 0]
                if len(negs) % 2 == 0:
                    best = max(best, len(seg))
                else:
                    best = max(best, len(seg) - negs[0] - 1, negs[-1])
                start = i + 1
        return best

print([SolutionSegments().getMaxLen(x) for x, _ in examples])

[4, 3, 2]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho các lát cắt tạm (có thể đưa về `O(1)` bằng cách chỉ lưu chỉ số).

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 3: `[-1, -2, -3, 0, 1]`.

| i | x | pos | neg | best | Giải thích |
|---|---|-----|-----|------|------------|
| 0 | −1 | 0 | 1 | 0 | `[-1]` âm |
| 1 | −2 | 2 | 1 | 2 | `[-1,-2]` dương; `[-2]` âm |
| 2 | −3 | 2 | 3 | 2 | `[-2,-3]` dương; `[-1,-2,-3]` âm |
| 3 | 0 | 0 | 0 | 2 | reset |
| 4 | 1 | 1 | 0 | 2 | `[1]` dương |

In [16]:
def trace_dp(nums):
    pos = neg = best = 0
    for i, x in enumerate(nums):
        old = (pos, neg)
        if x > 0:
            pos, neg = pos + 1, (neg + 1 if neg else 0)
            why = "dương: dương+1, âm+1 (nếu có đoạn âm)"
        elif x < 0:
            pos, neg = (neg + 1 if neg else 0), pos + 1
            why = "âm: đổi vai (pos <- neg+1, neg <- pos+1)"
        else:
            pos = neg = 0
            why = "số 0: reset"
        best = max(best, pos)
        print(f"i={i}, x={x:>2}: (pos,neg) {old} -> ({pos},{neg}), best={best} | {why}")
    return best

trace_dp([-1, -2, -3, 0, 1])

i=0, x=-1: (pos,neg) (0, 0) -> (0,1), best=0 | âm: đổi vai (pos <- neg+1, neg <- pos+1)
i=1, x=-2: (pos,neg) (0, 1) -> (2,1), best=2 | âm: đổi vai (pos <- neg+1, neg <- pos+1)
i=2, x=-3: (pos,neg) (2, 1) -> (2,3), best=2 | âm: đổi vai (pos <- neg+1, neg <- pos+1)
i=3, x= 0: (pos,neg) (2, 3) -> (0,0), best=2 | số 0: reset
i=4, x= 1: (pos,neg) (0, 0) -> (1,0), best=2 | dương: dương+1, âm+1 (nếu có đoạn âm)


2

In [17]:
trace_dp([1, -2, -3, 4])

i=0, x= 1: (pos,neg) (0, 0) -> (1,0), best=1 | dương: dương+1, âm+1 (nếu có đoạn âm)
i=1, x=-2: (pos,neg) (1, 0) -> (0,2), best=1 | âm: đổi vai (pos <- neg+1, neg <- pos+1)
i=2, x=-3: (pos,neg) (0, 2) -> (3,1), best=3 | âm: đổi vai (pos <- neg+1, neg <- pos+1)
i=3, x= 4: (pos,neg) (3, 1) -> (4,2), best=4 | dương: dương+1, âm+1 (nếu có đoạn âm)


4

Trace cách 4 — in `zero`, `first_neg`, `negs` và đoạn được chọn tại mỗi bước:

In [18]:
def trace_greedy(nums):
    best = 0
    zero, first_neg, negs = -1, None, 0
    for i, x in enumerate(nums):
        if x == 0:
            zero, first_neg, negs = i, None, 0
            print(f"i={i}: số 0 -> reset (zero={zero})")
            continue
        if x < 0:
            negs += 1
            if first_neg is None:
                first_neg = i
        if negs % 2 == 0:
            cand, seg = i - zero, (zero + 1, i)
        else:
            cand, seg = i - first_neg, (first_neg + 1, i)
        best = max(best, cand)
        print(f"i={i}: x={x:>2}, negs={negs}, first_neg={first_neg}, đoạn dương dài nhất kết thúc tại i = {seg if cand else 'rỗng'} (dài {cand}), best={best}")
    return best

trace_greedy([0, 1, -2, -3, -4])

i=0: số 0 -> reset (zero=0)
i=1: x= 1, negs=0, first_neg=None, đoạn dương dài nhất kết thúc tại i = (1, 1) (dài 1), best=1
i=2: x=-2, negs=1, first_neg=2, đoạn dương dài nhất kết thúc tại i = rỗng (dài 0), best=1
i=3: x=-3, negs=2, first_neg=2, đoạn dương dài nhất kết thúc tại i = (1, 3) (dài 3), best=3
i=4: x=-4, negs=3, first_neg=2, đoạn dương dài nhất kết thúc tại i = (3, 4) (dài 2), best=3


3

## 9. Code nộp LeetCode

In [19]:
class Solution:
    def getMaxLen(self, nums: List[int]) -> int:
        pos = neg = best = 0          # độ dài đoạn dài nhất kết thúc tại vị trí hiện tại có tích dương / âm
        for x in nums:
            if x > 0:
                pos, neg = pos + 1, (neg + 1 if neg else 0)
            elif x < 0:
                pos, neg = (neg + 1 if neg else 0), pos + 1
            else:
                pos = neg = 0
            best = max(best, pos)
        return best

for nums, want in examples:
    got = Solution().getMaxLen(nums)
    print(got, want)
    assert got == want

4 4
3 3
2 2


## 10. Vì sao thuật toán đúng

**Bổ đề (đúng đắn của chuyển trạng thái).** Gọi `P(i)`, `N(i)` là độ dài lớn nhất của đoạn **kết thúc tại `i`** có tích dương / âm (0 nếu không có). Mọi đoạn kết thúc tại `i` hoặc là `[i]`, hoặc là một đoạn kết thúc tại `i − 1` nối thêm `nums[i]`.
- `nums[i] > 0`: dấu không đổi khi nối. Đoạn dương kết thúc tại `i`: `[i]` (dài 1) hoặc đoạn dương tại `i−1` nối thêm ⇒ `P(i) = P(i−1) + 1` (vì `P(i−1) + 1 ≥ 1`). Đoạn âm kết thúc tại `i`: chỉ có thể là đoạn âm tại `i−1` nối thêm ⇒ `N(i) = N(i−1) + 1` nếu `N(i−1) > 0`, ngược lại `0`.
- `nums[i] < 0`: dấu đổi. Đoạn dương tại `i` = đoạn âm tại `i−1` nối thêm ⇒ `P(i) = N(i−1) + 1` nếu `N(i−1) > 0`, ngược lại `0`. Đoạn âm tại `i`: `[i]` hoặc đoạn dương tại `i−1` nối thêm ⇒ `N(i) = P(i−1) + 1`.
- `nums[i] = 0`: không đoạn nào kết thúc tại `i` có tích khác 0 ⇒ `P(i) = N(i) = 0`.

Mọi đoạn có tích dương đều kết thúc tại một `i` nào đó ⇒ đáp án `= max_i P(i)`.

**Cách tham lam.** Trong vùng không chứa `0`, tích đoạn `(a, i]` dương ⇔ số âm trong đó chẵn. Với `i` cố định, đoạn dài nhất là chọn `a` nhỏ nhất: `a = zero` nếu số âm từ `zero+1` tới `i` chẵn; ngược lại phải loại đúng một số âm lẻ ở đầu ⇒ `a = first_neg`. Đó chính là `P(i)`.

In [20]:
# thực nghiệm: P(i) của DP == đoạn dương dài nhất kết thúc tại i (vét cạn)
rng = random.Random(10)
for _ in range(300):
    nums = [rng.choice([-2, -1, 0, 1, 3]) for _ in range(rng.randint(1, 15))]
    pos = neg = 0
    for i, x in enumerate(nums):
        if x > 0:
            pos, neg = pos + 1, (neg + 1 if neg else 0)
        elif x < 0:
            pos, neg = (neg + 1 if neg else 0), pos + 1
        else:
            pos = neg = 0
        best_end_i = 0
        s = 1
        for l in range(i, -1, -1):
            if nums[l] == 0:
                break
            if nums[l] < 0:
                s = -s
            if s > 0:
                best_end_i = i - l + 1
        assert pos == best_end_i
print("P(i) đúng ở mọi vị trí: OK")

P(i) đúng ở mọi vị trí: OK


## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space |
|------|------|-------|
| Mọi đoạn con | `O(n²)` | `O(1)` |
| DP mảng | `O(n)` | `O(n)` |
| DP hai biến | `O(n)` | `O(1)` |
| Số 0 gần nhất + số âm đầu tiên | `O(n)` | `O(1)` |
| Chia đoạn theo 0 | `O(n)` | `O(n)` (lát cắt tạm) |

Không phụ thuộc độ lớn giá trị (chỉ dùng dấu).

In [21]:
rng = random.Random(1567)
big = [rng.choice([-10**9, -1, 0, 1, 10**9]) if rng.random() < 0.001 else rng.choice([-7, 3, 5]) for _ in range(10**5)]
for name, S in [("DPArray", SolutionDPArray), ("DP", SolutionDP), ("Greedy", SolutionGreedy), ("Segments", SolutionSegments), ("Solution", Solution)]:
    t0 = time.perf_counter()
    r = S().getMaxLen(big)
    print(f"{name:9s}: {r} ({time.perf_counter() - t0:.3f}s)")

DPArray  : 24022 (0.021s)
DP       : 24022 (0.011s)
Greedy   : 24022 (0.012s)
Segments : 24022 (0.010s)
Solution : 24022 (0.009s)


## 12. Bộ test

In [22]:
METHODS = {
    "brute_force": brute_force,
    "dp_arrays": dp_arrays,
    "SolutionDPArray": lambda a: SolutionDPArray().getMaxLen(a),
    "SolutionDP": lambda a: SolutionDP().getMaxLen(a),
    "SolutionGreedy": lambda a: SolutionGreedy().getMaxLen(a),
    "SolutionSegments": lambda a: SolutionSegments().getMaxLen(a),
    "Solution": lambda a: Solution().getMaxLen(a),
}
for nums, want in examples:
    for name, f in METHODS.items():
        assert f(nums) == want, name
print("ví dụ đề: OK")

ví dụ đề: OK


In [23]:
# edge case
edge = [
    ([5], 1),
    ([-5], 0),                      # một số âm: không có đoạn dương
    ([0], 0),
    ([0, 0, 0], 0),
    ([-1, -1], 2),
    ([-1, 0, -1], 0),
    ([1, 2, 3, -1, -2, -3, 4], 5),  # bỏ hậu tố tốt hơn bỏ tiền tố
    ([1, -2, 3, 4, 5], 3),          # bỏ tiền tố [1,-2]
    ([10**9, -10**9, -10**9], 3),
    ([-1, 2], 1),
    ([1, -1], 1),
]
for nums, want in edge:
    for name, f in METHODS.items():
        assert f(nums) == want, (name, nums, f(nums))
print("edge case: OK")

edge case: OK


In [24]:
# ngẫu nhiên có seed: đối chiếu với vét cạn
rng = random.Random(2016)
for _ in range(2000):
    n = rng.randint(1, 30)
    pool = rng.choice([[-1, 1], [-2, -1, 0, 1, 2], [-10**9, 0, 10**9], [-3, 5, 7]])
    nums = [rng.choice(pool) for _ in range(n)]
    ref = brute_force(nums)
    for name, f in METHODS.items():
        assert f(nums) == ref, (name, nums)
print("2000 test ngẫu nhiên (seed 2016): mọi cách khớp vét cạn")

2000 test ngẫu nhiên (seed 2016): mọi cách khớp vét cạn


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cập nhật `pos`, `neg` tuần tự (không đồng thời)

Khi `x < 0`, viết `pos = ...` rồi `neg = pos + 1` ⇒ `neg` dùng `pos` **mới**.

In [25]:
def wrong_sequential(nums):
    pos = neg = best = 0
    for x in nums:
        if x > 0:
            pos += 1
            neg = neg + 1 if neg else 0
        elif x < 0:
            pos = neg + 1 if neg else 0
            neg = pos + 1                    # SAI: pos đã bị ghi đè
        else:
            pos = neg = 0
        best = max(best, pos)
    return best

for nums, want in examples + [([2, -1, -1], 3)]:
    print(nums, "| sai:", wrong_sequential(nums), "| đúng:", want)

[1, -2, -3, 4] | sai: 3 | đúng: 4
[0, 1, -2, -3, -4] | sai: 4 | đúng: 3
[-1, -2, -3, 0, 1] | sai: 4 | đúng: 2
[2, -1, -1] | sai: 2 | đúng: 3


Sửa: gán đồng thời `pos, neg = (neg + 1 if neg else 0), pos + 1` (hoặc lưu biến tạm).

### Lỗi 2 — Tăng `neg` khi chưa có đoạn âm nào

Với `x > 0`, viết `neg = neg + 1` vô điều kiện ⇒ tạo ra "đoạn âm" không tồn tại (ví dụ `[1]` bị coi là âm dài 1).

In [26]:
def wrong_neg_plus_one(nums):
    pos = neg = best = 0
    for x in nums:
        if x > 0:
            pos, neg = pos + 1, neg + 1      # SAI: neg chỉ tăng khi neg > 0
        elif x < 0:
            pos, neg = (neg + 1 if neg else 0), pos + 1
        else:
            pos = neg = 0
        best = max(best, pos)
    return best

for nums in [[1, -1], [3, 2, -5], [1, -2, -3, 4]]:
    print(nums, "| sai:", wrong_neg_plus_one(nums), "| đúng:", Solution().getMaxLen(nums))

[1, -1] | sai: 2 | đúng: 1
[3, 2, -5] | sai: 3 | đúng: 2
[1, -2, -3, 4] | sai: 4 | đúng: 4


Sửa: `neg = neg + 1 if neg else 0`; tương tự `pos = neg + 1 if neg else 0` khi `x < 0`.

### Lỗi 3 — Quên reset khi gặp số 0

Coi `0` như số dương (hoặc bỏ qua) ⇒ đoạn "vượt qua" số 0.

In [27]:
def wrong_no_zero_reset(nums):
    pos = neg = best = 0
    for x in nums:
        if x >= 0:                           # SAI: 0 bị xử lý như số dương
            pos, neg = pos + 1, (neg + 1 if neg else 0)
        else:
            pos, neg = (neg + 1 if neg else 0), pos + 1
        best = max(best, pos)
    return best

for nums, want in examples:
    print(nums, "| sai:", wrong_no_zero_reset(nums), "| đúng:", want)

[1, -2, -3, 4] | sai: 4 | đúng: 4
[0, 1, -2, -3, -4] | sai: 4 | đúng: 3
[-1, -2, -3, 0, 1] | sai: 4 | đúng: 2


Sửa: nhánh riêng `else: pos = neg = 0` cho `x == 0`.

### Lỗi 4 — Chia đoạn theo 0 nhưng khi số âm lẻ chỉ bỏ tiền tố

Quên lựa chọn "bỏ hậu tố từ số âm cuối cùng".

In [28]:
def wrong_prefix_only(nums):
    best = 0
    seg = []
    for x in nums + [0]:
        if x == 0:
            negs = [k for k, y in enumerate(seg) if y < 0]
            if len(negs) % 2 == 0:
                best = max(best, len(seg))
            else:
                best = max(best, len(seg) - negs[0] - 1)     # SAI: thiếu phương án negs[-1]
            seg = []
        else:
            seg.append(x)
    return best

for nums in [[1, 2, 3, -1, -2, -3, 4], [0, 1, -2, -3, -4], [5, 6, -1]]:
    print(nums, "| sai:", wrong_prefix_only(nums), "| đúng:", Solution().getMaxLen(nums))

[1, 2, 3, -1, -2, -3, 4] | sai: 3 | đúng: 5
[0, 1, -2, -3, -4] | sai: 2 | đúng: 3
[5, 6, -1] | sai: 0 | đúng: 2


Sửa: `max(len(seg) − negs[0] − 1, negs[-1])`.

### Lỗi 5 — Khởi tạo đáp án bằng 1 (nghĩ "ít nhất một phần tử")

Mảng toàn số âm đơn lẻ hoặc toàn 0 có đáp án `0`.

In [29]:
def wrong_init_one(nums):
    return max(1, Solution().getMaxLen(nums))    # SAI: đáp án có thể là 0

for nums in [[-5], [0, 0], [-1, 0, -1]]:
    print(nums, "| sai:", wrong_init_one(nums), "| đúng:", Solution().getMaxLen(nums))

[-5] | sai: 1 | đúng: 0
[0, 0] | sai: 1 | đúng: 0
[-1, 0, -1] | sai: 1 | đúng: 0


Sửa: khởi tạo `best = 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài về **tích** đoạn con ⇒ nghĩ tới **dấu** và **số 0** trước; số âm đổi vai trò lớn/nhỏ ⇒ cần **hai trạng thái** (giống LC 152 giữ cả max và min).
- "Đoạn con dài nhất/tốt nhất" ⇒ trạng thái "kết thúc tại `i`" + max toàn cục.
- Điều kiện chẵn/lẻ trên đoạn ⇒ có thể dùng **tiền tố chẵn lẻ**: tích `(a, i]` dương ⇔ parity số âm tại `a` và `i` bằng nhau ⇒ lưu chỉ số **đầu tiên** của mỗi parity (chính là cách 4).
- Gán tuple để cập nhật đồng thời nhiều trạng thái.

## 15. Biến thể của bài

1. **Maximum Product Subarray (LC 152)**: giá trị tích lớn nhất — giữ `cur_max`, `cur_min`, số âm hoán đổi hai giá trị.
2. **Đoạn dài nhất có tích âm** (bài tập 5) — đáp án `max neg`.
3. **Đếm số đoạn con có tích dương** — dùng tiền tố parity: đếm cặp cùng parity trong mỗi vùng không có 0.
4. **Tích dương và độ dài ≤ K** hoặc **xoá tối đa một phần tử** — mở rộng trạng thái DP.
5. **Subarray Product Less Than K (LC 713)** — số dương, cửa sổ trượt.

In [30]:
# Biến thể 1 (LC 152): giá trị tích lớn nhất của đoạn con
def max_product(nums):
    best = cur_max = cur_min = nums[0]
    for x in nums[1:]:
        if x < 0:
            cur_max, cur_min = cur_min, cur_max
        cur_max = max(x, cur_max * x)
        cur_min = min(x, cur_min * x)
        best = max(best, cur_max)
    return best

print(max_product([2, 3, -2, 4]), max_product([-2, 0, -1]), "(LC 152: 6 và 0)")

6 0 (LC 152: 6 và 0)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Vét cạn | mọi đoạn, theo dõi dấu | `O(n²)` | `O(1)` | kiểm chứng |
| 2. DP mảng | `pos[i]`, `neg[i]` | `O(n)` | `O(n)` | học / in bảng |
| 3. DP hai biến | như 2, chỉ giữ giá trị trước | `O(n)` | `O(1)` | **bản nộp** |
| 4. Số 0 gần nhất + âm đầu tiên | parity tiền tố | `O(n)` | `O(1)` | khi thích tư duy tham lam / chẵn lẻ |
| 5. Chia đoạn theo 0 | đếm số âm, bỏ đầu/đuôi | `O(n)` | `O(n)` | giải thích bằng lời dễ nhất |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `getMaxLen([-1])`, `getMaxLen([0, 0])`, `getMaxLen([1, 2, 3])`?

**Bài 2 (tính tay):** Lập bảng `pos`, `neg` cho `nums = [-1, 2, -3, -4, 0, 5, -6]` và cho đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_step(pos, neg, x)` trả về cặp `(pos', neg')` sau khi thêm phần tử `x`.

**Bài 4 (tự viết lại):** Viết `ex4_best_segment(nums)` trả về cặp chỉ số `(l, r)` của **một** đoạn dài nhất có tích dương, hoặc `None` nếu không có.

**Bài 5 (mini challenge):** Viết `ex5_max_len_negative(nums)` — độ dài lớn nhất của đoạn con có tích **âm**.

In [31]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_step(pos, neg, x):
    if x > 0:
        # TODO
        pass
    elif x < 0:
        # TODO
        pass
    else:
        # TODO
        pass
    return pos, neg

In [32]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_best_segment(nums):
    # TODO
    raise NotImplementedError

def ex5_max_len_negative(nums):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một số âm đứng một mình có tích âm. (2) Số 0 có tích 0. (3) `0`, `0`, `3`.

**Bài 2.** (1) Bắt đầu `(pos,neg) = (0,1)` tại `-1`. (2) Tiếp `2` → `(1,2)`; `-3` → `(3,2)`; `-4` → `(3,4)`; `0` → `(0,0)`; `5` → `(1,0)`; `-6` → `(0,2)`. (3) Đáp án `3` (`[-1,2,-3]` hoặc `[2,-3,-4]`).

**Bài 3.** (1) `x > 0`: `pos + 1`, `neg + 1 if neg else 0`. (2) `x < 0`: `neg + 1 if neg else 0`, `pos + 1`. (3) `x == 0`: `0, 0`.

**Bài 4.** (1) Dùng cách 4: với mỗi `i`, đoạn dương dài nhất kết thúc tại `i` bắt đầu tại `zero + 1` hoặc `first_neg + 1`. (2) Ghi lại `(l, i)` khi cải thiện `best`. (3) Không cải thiện lần nào ⇒ `None`.

**Bài 5.** (1) Giữ nguyên DP. (2) Đáp án là `max neg`. (3) Hoặc tham lam: số âm lẻ ⇒ cả đoạn `(zero, i]`; chẵn và ≥ 2 ⇒ `(first_neg, i]`.

## 19. Đáp án tham khảo cho bài tập

In [33]:
# Bài 1
a1 = (Solution().getMaxLen([-1]), Solution().getMaxLen([0, 0]), Solution().getMaxLen([1, 2, 3]))
print("Bài 1:", a1)
assert a1 == (0, 0, 3)

# Bài 2
a2 = trace_dp([-1, 2, -3, -4, 0, 5, -6])
assert a2 == 3 == brute_force([-1, 2, -3, -4, 0, 5, -6])

Bài 1: (0, 0, 3)
i=0, x=-1: (pos,neg) (0, 0) -> (0,1), best=0 | âm: đổi vai (pos <- neg+1, neg <- pos+1)
i=1, x= 2: (pos,neg) (0, 1) -> (1,2), best=1 | dương: dương+1, âm+1 (nếu có đoạn âm)
i=2, x=-3: (pos,neg) (1, 2) -> (3,2), best=3 | âm: đổi vai (pos <- neg+1, neg <- pos+1)
i=3, x=-4: (pos,neg) (3, 2) -> (3,4), best=3 | âm: đổi vai (pos <- neg+1, neg <- pos+1)
i=4, x= 0: (pos,neg) (3, 4) -> (0,0), best=3 | số 0: reset
i=5, x= 5: (pos,neg) (0, 0) -> (1,0), best=3 | dương: dương+1, âm+1 (nếu có đoạn âm)
i=6, x=-6: (pos,neg) (1, 0) -> (0,2), best=3 | âm: đổi vai (pos <- neg+1, neg <- pos+1)


In [34]:
# Bài 3
def ex3_step_sol(pos, neg, x):
    if x > 0:
        return pos + 1, (neg + 1 if neg else 0)
    if x < 0:
        return (neg + 1 if neg else 0), pos + 1
    return 0, 0

rng3 = random.Random(3)
for _ in range(500):
    nums = [rng3.choice([-2, -1, 0, 1, 2]) for _ in range(rng3.randint(1, 20))]
    pos = neg = best = 0
    for x in nums:
        pos, neg = ex3_step_sol(pos, neg, x)
        best = max(best, pos)
    assert best == brute_force(nums)
print("Bài 3: OK")

Bài 3: OK


In [35]:
# Bài 4
def ex4_best_segment_sol(nums):
    best, seg = 0, None
    zero, first_neg, negs = -1, None, 0
    for i, x in enumerate(nums):
        if x == 0:
            zero, first_neg, negs = i, None, 0
            continue
        if x < 0:
            negs += 1
            if first_neg is None:
                first_neg = i
        start = zero + 1 if negs % 2 == 0 else first_neg + 1
        if i - start + 1 > best:
            best, seg = i - start + 1, (start, i)
    return seg

rng4 = random.Random(4)
for _ in range(800):
    nums = [rng4.choice([-3, -1, 0, 2, 5]) for _ in range(rng4.randint(1, 20))]
    seg = ex4_best_segment_sol(nums)
    want = brute_force(nums)
    if seg is None:
        assert want == 0
    else:
        l, r = seg
        s = 1
        for x in nums[l:r + 1]:
            s *= (x > 0) - (x < 0)
        assert s == 1 and r - l + 1 == want
print("Bài 4: OK, ví dụ:", ex4_best_segment_sol([0, 1, -2, -3, -4]), ex4_best_segment_sol([-5]))

Bài 4: OK, ví dụ: (1, 3) None


In [36]:
# Bài 5
def ex5_max_len_negative_sol(nums):
    pos = neg = best = 0
    for x in nums:
        if x > 0:
            pos, neg = pos + 1, (neg + 1 if neg else 0)
        elif x < 0:
            pos, neg = (neg + 1 if neg else 0), pos + 1
        else:
            pos = neg = 0
        best = max(best, neg)
    return best

def ex5_brute(nums):
    n, best = len(nums), 0
    for l in range(n):
        s = 1
        for r in range(l, n):
            if nums[r] == 0:
                break
            if nums[r] < 0:
                s = -s
            if s < 0:
                best = max(best, r - l + 1)
    return best

rng5 = random.Random(5)
for _ in range(1000):
    nums = [rng5.choice([-4, -1, 0, 1, 6]) for _ in range(rng5.randint(1, 25))]
    assert ex5_max_len_negative_sol(nums) == ex5_brute(nums)
print("Bài 5: OK, ví dụ:", ex5_max_len_negative_sol([1, -2, -3, 4]), ex5_max_len_negative_sol([0, 1, -2, -3, -4]))

Bài 5: OK, ví dụ: 2 4


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def getMaxLen(self, nums):
        pos = neg = best = 0
        for x in nums:
            if x > 0:
                pos, neg = pos + 1, (neg + 1 if neg else 0)
            elif x < 0:
                pos, neg = (neg + 1 if neg else 0), pos + 1
            else:
                pos = neg = 0
            best = max(best, pos)
        return best
```

**Template "hai trạng thái dấu":** `pos`/`neg` (hoặc `max`/`min` với LC 152) cho đoạn kết thúc tại `i`; số âm hoán đổi vai trò; số 0 reset.

**Checklist trước khi submit:**
- [ ] Không tính tích thật — chỉ dùng dấu.
- [ ] Cập nhật `pos`, `neg` **đồng thời**.
- [ ] `neg + 1` / `pos ← neg + 1` chỉ khi `neg > 0`.
- [ ] Nhánh riêng cho `x == 0` (reset).
- [ ] `best` khởi tạo `0`.
- [ ] Thử nhanh: ba ví dụ → `4, 3, 2`; `[-5] → 0`; `[1,2,3,-1,-2,-3,4] → 5`.